# MAE 실습: 이미지 패치를 가리고 복원하기

patchify, masking, 간단한 복원 흐름을 직접 만듭니다.


In [ ]:
# numpy는 패치 조작, matplotlib은 마스킹 결과 시각화에 사용합니다.
import numpy as np
import matplotlib.pyplot as plt

# 마스킹은 무작위 과정이므로 시드를 고정합니다.
np.random.seed(55)


In [ ]:
# 16x16 장난감 이미지를 만듭니다.
image = np.zeros((16, 16))
image[2:7, 3:13] = 0.7
image[7:12, 6:10] = 1.0
image[12:14, 4:12] = 0.5
plt.imshow(image, cmap="gray", vmin=0, vmax=1)
plt.title("original image")
plt.axis("off")
plt.show()


In [ ]:
# 이미지를 패치로 나누고 다시 이미지로 조립하는 함수를 만듭니다.
def patchify(x, patch_size=4):
    # 각 패치는 flatten되어 한 줄 벡터가 됩니다.
    h, w = x.shape
    patches = []
    for row in range(0, h, patch_size):
        for col in range(0, w, patch_size):
            patches.append(x[row:row+patch_size, col:col+patch_size].reshape(-1))
    return np.array(patches)

def unpatchify(patches, image_shape=(16, 16), patch_size=4):
    # flatten된 패치들을 원래 이미지 격자 위치에 되돌립니다.
    h, w = image_shape
    output = np.zeros(image_shape)
    idx = 0
    for row in range(0, h, patch_size):
        for col in range(0, w, patch_size):
            output[row:row+patch_size, col:col+patch_size] = patches[idx].reshape(patch_size, patch_size)
            idx += 1
    return output

patches = patchify(image)
print("전체 패치 수:", len(patches))
print("패치 벡터 길이:", patches.shape[1])


In [ ]:
# MAE처럼 75% 패치를 가립니다.
mask_ratio = 0.75
num_patches = len(patches)
num_masked = int(num_patches * mask_ratio)
shuffled = np.random.permutation(num_patches)
masked_indices = shuffled[:num_masked]
visible_indices = shuffled[num_masked:]
visible_patches = patches[visible_indices]
print("encoder에 들어가는 패치 수:", len(visible_patches))
print("가려진 패치 수:", len(masked_indices))


In [ ]:
# 시각화를 위해 가려진 패치를 회색 값으로 채웁니다.
masked_patches = patches.copy()
masked_patches[masked_indices] = 0.25
masked_image = unpatchify(masked_patches, image_shape=image.shape)

# 단순 기준선: 보이는 패치 평균으로 가려진 패치를 채웁니다.
reconstructed_patches = masked_patches.copy()
reconstructed_patches[masked_indices] = visible_patches.mean(axis=0)
reconstructed = unpatchify(reconstructed_patches, image_shape=image.shape)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, data, title in zip(axes, [image, masked_image, reconstructed], ["original", "75% masked", "simple reconstruction"]):
    ax.imshow(data, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.show()


## 관찰 포인트

MAE는 라벨 없이도 만들 수 있는 복원 문제로 시각 표현을 학습합니다.
